# Batch LLM Processing with Live Progress

Run 100 queries through a mock async LLM API with:
- **Controlled concurrency** — `max_concurrency=10` keeps 10 requests in flight
- **Live progress bars** — `RichProgressProcessor` shows real-time completion
- **Checkpointing** — persist every result to SQLite, query later via Python or CLI
- **Error handling** — `error_handling="continue"` collects failures without stopping the batch

In [ ]:
import asyncio
import random

from hypergraph import AsyncRunner, Graph, node
from hypergraph.events.rich_progress import RichProgressProcessor

## 1. Define the graph

A simple 2-node pipeline: **classify** a query, then **generate** a response.
Both nodes simulate real LLM latency with `asyncio.sleep`.

In [ ]:
@node(output_name="category")
async def classify(query: str) -> str:
    """Classify a query into a category (mock LLM call)."""
    await asyncio.sleep(5)#random.uniform(5, 20))
    categories = ["question", "instruction", "creative", "analysis"]
    # Deterministic but varied: hash the query
    return categories[hash(query) % len(categories)]


@node(output_name="response")
async def generate(query: str, category: str) -> str:
    """Generate a response based on query and category (mock LLM call)."""
    await asyncio.sleep(random.uniform(0.1, 0.3))
    # Simulate occasional failures (5% rate)
    if random.random() < 0.05:
        raise TimeoutError(f"LLM timeout for: {query[:30]}")
    return f"[{category}] Response to: {query[:50]}"


llm_pipeline = Graph([classify, generate], name="llm-pipeline")

In [ ]:
llm_pipeline

## 2. Generate 100 queries

In [ ]:
topics = [
    "quantum computing", "climate change", "machine learning",
    "ancient history", "cooking techniques", "space exploration",
    "music theory", "philosophy", "biology", "economics",
]
templates = [
    "Explain {} in simple terms",
    "What are the latest advances in {}?",
    "Compare two approaches to {}",
    "Write a short essay about {}",
    "What are common misconceptions about {}?",
    "How does {} affect everyday life?",
    "Summarize the history of {}",
    "What are the ethical implications of {}?",
    "Predict the future of {}",
    "What should a beginner know about {}?",
]

queries = [t.format(topic) for topic in topics for t in templates]
print(f"{len(queries)} queries")
print(f"Sample: {queries[0]!r}")

## 3. Run the batch with live progress

This is the main event: `runner.map()` fans out 100 queries across 10 concurrent workers.
The `RichProgressProcessor` renders live progress bars as each item completes.

In [ ]:
progress = RichProgressProcessor(transient=False)
runner = AsyncRunner()

results = await runner.map(
    llm_pipeline,
    {"query": queries},
    map_over="query",
    #max_concurrency=10000,
    error_handling="continue",
    event_processors=[progress],
)

In [ ]:
results

In [ ]:
# Aggregate log — batch overview
results.log

In [ ]:
# Collect all responses (None for failed items)
responses = results["response"]
succeeded = [r for r in responses if r is not None]
print(f"{len(succeeded)} succeeded, {len(responses) - len(succeeded)} failed")
print(f"\nSample responses:")
for r in succeeded[:5]:
    print(f"  {r}")

In [ ]:
# Which items failed?
if results.failed:
    for i, r in enumerate(results):
        if r.failed:
            print(f"  Item {i}: {r.error}")
else:
    print("No failures this run (5% random failure rate)")

In [ ]:
# Per-node stats across all 100 items
results.log.node_stats

## 5. With checkpointing — persist every result

Add a `SqliteCheckpointer` and `workflow_id` to persist the entire batch.
Each item becomes a child run you can query later.

In [ ]:
import tempfile, os
from hypergraph.checkpointers import SqliteCheckpointer

_tmp = tempfile.mkdtemp()
DB = os.path.join(_tmp, "llm_batch.db")

cp = SqliteCheckpointer(DB, durability="sync")
runner_cp = AsyncRunner(checkpointer=cp)

In [ ]:

results = await runner_cp.map(
    llm_pipeline,
    {"query": queries},
    map_over="query",
    max_concurrency=1000,
    error_handling="continue",
    event_processors=[RichProgressProcessor(transient=False)],
    workflow_id="llm-batch-001",
)

In [ ]:
print(f"\nDB: {DB}")

In [ ]:
cp

In [ ]:
# Parent batch run
cp.get_run("llm-batch-001")

In [ ]:
# Child runs (first 5)
children = cp.runs(parent_run_id="llm-batch-001")
print(f"{len(children)} child runs")
for child in children[:5]:
    print(f"  {child.id}: {child.status.value}")

In [ ]:
# Drill into a specific item's values
cp.values("llm-batch-001/0")

In [ ]:
# Stats across the batch — how long did classify vs generate take?
cp.stats("llm-batch-001/0")

## 6. CLI inspection (from another cell or terminal)

The same data is accessible via `hypergraph runs` — useful for post-hoc debugging.

In [ ]:
!hypergraph runs --db {DB}

In [ ]:
# List child runs of the batch
!hypergraph runs ls --parent llm-batch-001 --db {DB}

In [ ]:
# Inspect a single item
!hypergraph runs show llm-batch-001/0 --db {DB}

In [ ]:
# Find all timeouts
!hypergraph runs search "timeout" --db {DB}